In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
import os
import PIL

In [3]:
path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/homer_bart_2.zip'
zip_objetict = zipfile.ZipFile(file = path, mode = 'r')
zip_objetict.extractall('./')
zip_objetict.close()

In [4]:
batch_size = 32
img_height = 180
img_width = 180

In [5]:
data_train = '/content/homer_bart_2/training_set'
data_test = '/content/homer_bart_2/test_set'

In [6]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_train,
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 215 files belonging to 2 classes.


In [7]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_test,
  image_size=(img_height, img_width),
  batch_size=batch_size)

Found 54 files belonging to 2 classes.


In [8]:
class_names = train_ds.class_names
print(class_names)

['bart', 'homer']


In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [10]:
for image_batch, labels_batch in train_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

(32, 180, 180, 3)
(32,)


In [11]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [12]:
normalization_layer = layers.Rescaling(1./255)

In [13]:
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
print(np.min(first_image), np.max(first_image))

0.0 1.0


In [14]:
num_classes = len(class_names)

model = Sequential([
  keras.Sequential(
  [
    layers.RandomFlip("horizontal",
                      input_shape=(img_height,
                                  img_width,
                                  3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
  ]
),
  layers.Rescaling(1./255),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.2),
  layers.Dense(256, activation='relu'),
  layers.Dense(num_classes)
])

In [15]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
epochs = 100
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

In [20]:
modelo = model.to_json()
with open('modelo.json', 'w') as json_file:
  json_file.write(modelo)

In [21]:
from keras.models import save_model
model_save = save_model(model, 'weights.hdf5')

<ipython-input-21-1310051c93f8>:2: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  model_save = save_model(model, 'weights.hdf5')


In [22]:
with open('modelo.json') as json_file:
  json_saved_model = json_file.read()
json_saved_model

'{"class_name": "Sequential", "config": {"name": "sequential_1", "layers": [{"module": "keras.layers", "class_name": "InputLayer", "config": {"batch_input_shape": [null, 180, 180, 3], "dtype": "float32", "sparse": false, "ragged": false, "name": "sequential_input"}, "registered_name": null}, {"module": "keras", "class_name": "Sequential", "config": {"name": "sequential", "layers": [{"module": "keras.layers", "class_name": "InputLayer", "config": {"batch_input_shape": [null, 180, 180, 3], "dtype": "float32", "sparse": false, "ragged": false, "name": "random_flip_input"}, "registered_name": null}, {"module": "keras.layers", "class_name": "RandomFlip", "config": {"name": "random_flip", "trainable": true, "dtype": "float32", "batch_input_shape": [null, 180, 180, 3], "mode": "horizontal", "seed": null}, "registered_name": null, "build_config": {"input_shape": [null, 180, 180, 3]}}, {"module": "keras.layers", "class_name": "RandomRotation", "config": {"name": "random_rotation", "trainable": 

In [23]:
model_loaded = tf.keras.models.model_from_json(json_saved_model)
model_loaded.load_weights('/content/weights.hdf5')
model_loaded.compile(loss = 'binary_crossentropy',
                     optimizer = 'adam',
                     metrics = ['accuracy'])

In [25]:
model_loaded.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential (Sequential)     (None, 180, 180, 3)       0         
                                                                 
 rescaling_1 (Rescaling)     (None, 180, 180, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 180, 180, 64)      1792      
                                                                 
 max_pooling2d (MaxPooling2  (None, 90, 90, 64)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 90, 90, 64)        36928     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 45, 45, 64)        0         
 g2D)                                                 

In [43]:
imagem = cv2.imread('/content/homer_bart_2/test_set/bart/bart12.bmp')
imagem = cv2.resize(imagem, (180, 180))
imagem = imagem / 255
imagem = imagem.reshape(-1, 180, 180, 3)

In [44]:
pred = model_loaded.predict(imagem)
pred = np.argmax(pred)
if pred == 0:
  print('Bart')
else:
  print('Homer')

1/1 [==============================] - 0s 25ms/step
Homer
